# 04 — Prepare SOLWEIG raster inputs

SOLWEIG requires projected raster surfaces. This notebook converts the enriched VoxCity model into:

- `dsm.tif` — digital surface model (terrain + buildings);
- `cdsm.tif` — canopy surface;
- `dem.tif` — terrain elevation.

All rasters are reprojected to EPSG:25830 at approximately 5 m resolution.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio

from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from voxcity.generator import load_voxcity

SOLWEIG_INPUTS = DATA_INTERIM / "solweig"
SOLWEIG_INPUTS.mkdir(parents=True, exist_ok=True)

vox = load_voxcity(
    DATA_INTERIM / "sevilla_voxcity_eubucco.pkl",
    trusted=True,
)

## 1. Extract the surface arrays

In [ ]:
building_height = vox.buildings.heights
canopy_top = vox.tree_canopy.top
dem = vox.dem.elevation

dsm = dem + building_height
cdsm = canopy_top.copy()

print("Grid shape:", dsm.shape)
print("DSM range:", float(np.nanmin(dsm)), "to", float(np.nanmax(dsm)))

## 2. Define source and target georeferencing

In [ ]:
minlon, minlat, maxlon, maxlat = vox.voxels.meta.bounds
height, width = dsm.shape

src_crs = "EPSG:4326"
dst_crs = "EPSG:25830"

src_transform = from_bounds(
    minlon,
    minlat,
    maxlon,
    maxlat,
    width,
    height,
)

dst_transform, dst_width, dst_height = calculate_default_transform(
    src_crs,
    dst_crs,
    width,
    height,
    minlon,
    minlat,
    maxlon,
    maxlat,
    resolution=5,
)

print("Projected raster size:", dst_height, "×", dst_width)

## 3. Reproject arrays to the projected 5 m grid

In [ ]:
def reproject_to_5m(array):
    destination = np.zeros(
        (dst_height, dst_width),
        dtype=np.float32,
    )

    # VoxCity array row order is opposite rasterio's north-up convention.
    source = np.flipud(array).astype(np.float32)

    reproject(
        source=source,
        destination=destination,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )

    return destination


dsm_5m = reproject_to_5m(dsm)
cdsm_5m = reproject_to_5m(cdsm)
dem_5m = reproject_to_5m(dem)

## 4. Write GeoTIFFs

In [ ]:
def write_projected_geotiff(path, array):
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype="float32",
        crs=dst_crs,
        transform=dst_transform,
        compress="lzw",
    ) as dst:
        dst.write(array.astype("float32"), 1)


write_projected_geotiff(SOLWEIG_INPUTS / "dsm.tif", dsm_5m)
write_projected_geotiff(SOLWEIG_INPUTS / "cdsm.tif", cdsm_5m)
write_projected_geotiff(SOLWEIG_INPUTS / "dem.tif", dem_5m)

## 5. Validate the written rasters

In [ ]:
for name in ["dsm.tif", "cdsm.tif", "dem.tif"]:
    path = SOLWEIG_INPUTS / name

    with rasterio.open(path) as src:
        array = src.read(1)

        print(
            name,
            "| CRS:", src.crs,
            "| shape:", src.shape,
            "| resolution:", src.res,
            "| min/mean/max:",
            round(float(np.nanmin(array)), 2),
            round(float(np.nanmean(array)), 2),
            round(float(np.nanmax(array)), 2),
        )